# M08/M09 single-model training and benchmark

Train the registered ConvNeXt-Tiny (M08) and RAD-DINO (M09) teacher models on NIH ChestX-ray14 using the study's seed-42 configuration. This notebook contains training and forward-pass benchmarking only; it does not run post-training analyses or compute test metrics.

For the registered M10 distillation experiment, use the retained, hash-pinned teacher bundle in `single_model/output/results_s42/`. Freshly trained checkpoints are new experiment artifacts and will not automatically satisfy the registered M10 hash checks.


In [ ]:
# ============================ CONFIG ============================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")      # determinism; set before CUDA starts
import json, glob, time, math, random, hashlib, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ---- what to run in THIS session ----
RUN_CNN      = False    # Session A (GPU): train ConvNeXt-Tiny, export logits
RUN_VIT      = False    # Session B (GPU): train RAD-DINO, export logits
RUN_BENCH    = False    # Session T (GPU): forward-pass latency of both models

SEED = 42               # the only training seed of this study

PROTOCOL_VERSION = "2.0.0"
OUT_DIR   = os.environ.get("CXR_OUT_DIR", "/kaggle/working/single_model/output/results_s42")
RUNS_DIR  = os.path.join(OUT_DIR, "runs")
INPUT_DIR = os.environ.get("CXR_INPUT_DIR", "/kaggle/input")
DATA_ROOT = "/kaggle/input/datasets/nih-chest-xrays/data"      # NIH images + Data_Entry_2017.csv (sessions A/B)
NUM_WORKERS = 4

DATASET_ID, SPLIT_ID = "NIH_CXR14", "NIH_OFFICIAL_PV"
VAL_FRACTION, SPLIT_SEED = 0.15, 42                              # validation = 15 % of train_val patients
EXPECTED_COUNTS = {"train_val": 86524, "test": 25596, "total": 112120}
EXPECTED_LIST_SHA256 = {                                          # SHA-256 of sorted, LF-joined image ids
    "train_val_list.txt": "61fbe896321c1c1c8b75f3e4f3a08e4fef6486d95ef8a667c31d4d60dca6cb81",
    "test_list.txt":      "38ca5ef7f756092946f57c1a59faca882ed589a1ab1f72590b45dc06c6d5e1cc"}
EXPECTED_SPLIT_SHA256 = "b9f2fbc9aa061dafa76994494b92779651dc3ddd2ed3e7b82c70bbf5cf7071dd"   # split of the seed-42 runs

CLASSES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
           "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"]
C = len(CLASSES)

ASL = dict(gamma_neg=4, gamma_pos=1, clip=0.05)
CNN_CFG = dict(
    kind="cnn", model_id="M08_CONVNEXT_TINY_ASL_384_CR", timm_name="convnext_tiny.fb_in22k_ft_in1k",
    img_size=384, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225),
    batch_size=32, warmup_epochs=1, finetune_epochs=8, patience=2,
    lr_head_warmup=1e-3, lr_head=1e-4, lr_backbone=1e-4,
    weight_decay=0.05, drop_path=0.1, use_ema=False, ema_decay=0.999, grad_clip_norm=1.0, asl=ASL, time_budget_h=8.0)
VIT_CFG = dict(
    kind="raddino", model_id="M09_RAD_DINO_518_CR", hf_name="microsoft/rad-dino",
    img_size=518, mean=(0.5307,) * 3, std=(0.2583,) * 3,
    batch_size=12, warmup_epochs=2, finetune_epochs=4, patience=2, unfreeze_last=8, layer_decay=0.8,
    lr_head_warmup=1e-3, lr_head=2e-4, lr_backbone=2e-5,
    weight_decay=0.05, drop_path=0.1, use_ema=True, ema_decay=0.999, grad_clip_norm=1.0, asl=ASL, time_budget_h=9.0)

# ---- timing benchmark (session T) ----
LATENCY_BATCH_SIZES = [1, 32]
WARMUP_ITERS, TIMED_ITERS, TIMING_REPEATS = 50, 200, 3

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# ---------------- helpers ----------------
def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def write_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def save_table(df_, name):
    """CSV, UTF-8, full precision (rounding only when displayed)."""
    df_.to_csv(f"{OUT_DIR}/{name}.csv", index=False, float_format="%.17g", na_rep="")

def fmt(x, d=4):
    return "n/a" if x is None or not np.isfinite(x) else f"{np.round(x, d):.{d}f}"

def _walk(bases=(OUT_DIR, INPUT_DIR)):
    """os.walk over outputs and inputs, skipping the 112k-image folders."""
    for base in dict.fromkeys(bases):
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith("images")]
            yield root, dirs, files

def find_file(fname):
    for root, _, files in _walk():
        if fname in files:
            return os.path.join(root, fname)
    return None

def run_folder_name(model_id):
    return f"{DATASET_ID}__{SPLIT_ID}__{model_id}__s{SEED}"

def find_run_dir(model_id):
    """Seed-42 run folder of a model that contains exported logits. Accepts the older names that carry a
    config hash (â€¦__{model_id}__{hash}__s42__r0). More than one distinct folder is an error."""
    hits = {}
    for root, dirs, files in _walk():
        name = os.path.basename(root)
        if ("logits_test.npz" in files and name.startswith(f"{DATASET_ID}__{SPLIT_ID}__{model_id}__")
                and (name.endswith(f"__s{SEED}") or f"__s{SEED}__" in name) and "probe" not in name):
            hits.setdefault(name, root)                    # OUT_DIR is searched first
    assert len(hits) <= 1, f"several seed-{SEED} folders for {model_id}: {sorted(hits)} - attach only one"
    return next(iter(hits.values()), None)

## Dataset and split checks

Attach the NIH ChestX-ray14 Kaggle dataset. The notebook verifies the official train/validation and test lists, image counts, split hash, and patient separation before training. It writes the split CSV and dataset summary to the output root. A mismatch stops the run instead of silently changing the experiment.


In [ ]:
NEED_DATA = RUN_CNN or RUN_VIT
if NEED_DATA:
    from sklearn.model_selection import GroupShuffleSplit
    meta_csv = os.path.join(DATA_ROOT, "Data_Entry_2017.csv")
    if not os.path.exists(meta_csv):
        meta_csv = next((os.path.join(r, f) for r, _, fs in os.walk(INPUT_DIR) for f in fs if f == "Data_Entry_2017.csv"), meta_csv)
    assert os.path.exists(meta_csv), "Data_Entry_2017.csv not found - check DATA_ROOT"
    DATA_ROOT = os.path.dirname(meta_csv)
    png = glob.glob(os.path.join(DATA_ROOT, "images_*", "images", "*.png")) or \
          [os.path.join(r, f) for r, _, fs in os.walk(DATA_ROOT) for f in fs if f.endswith(".png")]
    df = pd.read_csv(meta_csv)
    df["path"] = df["Image Index"].map({os.path.basename(p): p for p in png})
    assert len(df) == EXPECTED_COUNTS["total"] and df.path.notna().all(), "images missing on disk"
    labels = df["Finding Labels"].str.split("|")
    for c in CLASSES:
        df[c] = labels.apply(lambda L, c=c: c in L).astype(np.float32)
    df = df.sort_values("Image Index").reset_index(drop=True)      # canonical order

    # ---- official list files (QC-01) ----
    def read_list(fname):
        p = os.path.join(DATA_ROOT, fname)
        p = p if os.path.exists(p) else find_file(fname)
        assert p, f"{fname} not found"
        return open(p).read().split()
    tv_ids, te_ids = read_list("train_val_list.txt"), read_list("test_list.txt")
    list_sha = {"train_val_list.txt": sha256_bytes("\n".join(sorted(tv_ids)).encode()),
                "test_list.txt": sha256_bytes("\n".join(sorted(te_ids)).encode())}
    assert (len(tv_ids), len(te_ids)) == (EXPECTED_COUNTS["train_val"], EXPECTED_COUNTS["test"]), "QC-01: list counts"
    assert list_sha == EXPECTED_LIST_SHA256, "QC-01: list files differ from the official lists"
    assert not set(tv_ids) & set(te_ids) and set(tv_ids) | set(te_ids) == set(df["Image Index"]), "QC-01: overlap/coverage"

    # ---- split (identical code to the one that produced the seed-42 runs) ----
    df["split"] = np.where(df["Image Index"].isin(set(te_ids)), "test", "trainval")
    tv_frame = df[df.split == "trainval"]
    _, va = next(GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SPLIT_SEED)
                 .split(tv_frame, groups=tv_frame["Patient ID"]))
    df.loc[tv_frame.index, "split"] = "train"
    df.loc[tv_frame.index[va], "split"] = "val"
    split_text = (df[["Image Index", "Patient ID", "split"]]
                  .rename(columns={"Image Index": "image_index", "Patient ID": "patient_id", "split": "partition"})
                  .sort_values("image_index").to_csv(index=False, lineterminator="\n"))
    SPLIT_SHA256 = sha256_bytes(split_text.encode("utf-8"))
    assert SPLIT_SHA256 == EXPECTED_SPLIT_SHA256, \
        f"split hash {SPLIT_SHA256[:12]} != {EXPECTED_SPLIT_SHA256[:12]}: library versions changed the validation carve-out"
    with open(f"{OUT_DIR}/split_{SPLIT_ID}.csv", "w", encoding="utf-8", newline="\n") as f:
        f.write(split_text)
    pats = {s: set(df.loc[df.split == s, "Patient ID"]) for s in ("train", "val", "test")}
    assert not (pats["train"] & pats["val"] or pats["train"] & pats["test"] or pats["val"] & pats["test"]), "QC-02: patient leakage"

    df_train, df_val, df_test = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))

    # ---- T1 dataset characteristics ----
    t1 = []
    for part, fr in (("train", df_train), ("val", df_val), ("test", df_test)):
        Y = fr[CLASSES].values
        base = dict(dataset_id=DATASET_ID, split_id=SPLIT_ID, partition=part, n_images=len(fr),
                    n_patients=fr["Patient ID"].nunique(), label_cardinality_mean=Y.sum(1).mean(),
                    n_all_negative_images=int((Y.sum(1) == 0).sum()),
                    images_per_patient_mean=len(fr) / fr["Patient ID"].nunique(),
                    view_PA_fraction=(fr["View Position"] == "PA").mean(), male_fraction=(fr["Patient Gender"] == "M").mean(),
                    age_median=fr["Patient Age"].median(), age_q25=fr["Patient Age"].quantile(.25),
                    age_q75=fr["Patient Age"].quantile(.75))
        t1.append(dict(base, label="ALL", n_positive=int(Y.sum()), prevalence=np.nan))
        t1 += [dict(base, label=c, n_positive=int(Y[:, i].sum()), prevalence=Y[:, i].mean()) for i, c in enumerate(CLASSES)]
    save_table(pd.DataFrame(t1), "T1_dataset_characteristics")
    print(f"split {SPLIT_ID} ({SPLIT_SHA256[:12]}) | train {len(df_train):,} | val {len(df_val):,} | test {len(df_test):,}")

## Model training and benchmark implementation

Training uses the fixed M08/M09 configurations above, seed 42, validation macro-AUROC for checkpoint selection and early stopping, and exports checkpoints plus validation/test logits without TTA. Test logits are saved for later evaluation; no test metric is used during training. Set `RUN_BENCH = True` to run the forward-pass latency benchmark separately.


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def set_determinism():
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

class CXRDataset(Dataset):
    def __init__(self, paths, y, cfg, train):
        self.paths, self.y, s = paths, y.astype(np.float32), cfg["img_size"]
        self.tfm = (T.Compose([T.RandomResizedCrop(s, scale=(0.85, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(0.5),
                               T.RandomApply([T.RandomRotation(7)], p=0.5), T.ColorJitter(brightness=0.1, contrast=0.1),
                               T.ToTensor()]) if train else
                    T.Compose([T.Resize((s, s), interpolation=T.InterpolationMode.BILINEAR), T.ToTensor()]))
        self.norm = T.Normalize(cfg["mean"], cfg["std"])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L")               # grayscale -> [0, 1] -> 3 channels
        return self.norm(self.tfm(img).repeat(3, 1, 1)), self.y[i]

def loader(frame, cfg, train, batch_size=None):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(CXRDataset(frame["path"].values, frame[CLASSES].values, cfg, train),
                      batch_size=batch_size or cfg["batch_size"], shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0, drop_last=False,
                      worker_init_fn=worker_init_fn, generator=g)

class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., 2021)."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits.float())
        p_neg = (p - self.clip).clamp(min=0) if self.clip > 0 else p
        loss_pos = y * torch.log(p.clamp(min=self.eps)) * (1 - p).pow(self.gp)
        loss_neg = (1 - y) * torch.log((1 - p_neg).clamp(min=self.eps)) * p_neg.pow(self.gn)
        return -(loss_pos + loss_neg).sum(1).mean()

class CNNClassifier(nn.Module):
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        import timm
        self.backbone = timm.create_model(cfg["timm_name"], pretrained=pretrained, num_classes=0, drop_path_rate=cfg["drop_path"])
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.backbone.num_features, C))

    def forward(self, x):
        return self.head(self.backbone(x))

    def backbone_param_groups(self, cfg):
        return [{"params": list(self.backbone.parameters()), "lr": cfg["lr_backbone"]}]

class RadDinoClassifier(nn.Module):
    """RAD-DINO ViT-B/14 + dual pooling (CLS âŠ• mean patch = 1536-d)."""
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        from transformers import AutoModel, AutoConfig
        self.backbone = (AutoModel.from_pretrained(cfg["hf_name"], drop_path_rate=cfg["drop_path"]) if pretrained else
                         AutoModel.from_config(AutoConfig.from_pretrained(cfg["hf_name"], drop_path_rate=cfg["drop_path"])))
        self.n_reg = getattr(self.backbone.config, "num_register_tokens", 0)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(2 * self.backbone.config.hidden_size, C))

    def forward(self, x):
        h = self.backbone(pixel_values=x).last_hidden_state
        return self.head(torch.cat([h[:, 0], h[:, 1 + self.n_reg:].mean(1)], dim=1))

    def backbone_param_groups(self, cfg):
        layers, k = self.backbone.encoder.layer, cfg["unfreeze_last"]
        n = len(layers)
        groups = [{"params": list(self.backbone.layernorm.parameters()), "lr": cfg["lr_backbone"]}]
        groups += [{"params": list(layers[i].parameters()), "lr": cfg["lr_backbone"] * cfg["layer_decay"] ** (n - 1 - i)}
                   for i in range(n - k, n)]
        return groups

def build_model(cfg, pretrained=True):
    seed_everything(SEED)                                          # seeded head initialisation
    return (CNNClassifier(cfg, pretrained) if cfg["kind"] == "cnn" else RadDinoClassifier(cfg, pretrained)).to(DEVICE)

@torch.no_grad()
def predict_logits(model, dl):
    model.eval()
    out = []
    for x, _ in dl:
        with autocast():
            out.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy().astype(np.float32)

def configure_phase(model, cfg, phase):
    """warmup = head only; finetune = head + (part of) the backbone."""
    for p in model.parameters():
        p.requires_grad = False
    groups = [{"params": list(model.head.parameters()), "lr": cfg["lr_head_warmup"] if phase == "warmup" else cfg["lr_head"]}]
    if phase == "finetune":
        groups += model.backbone_param_groups(cfg)
    for g in groups:
        for p in g["params"]:
            p.requires_grad = True
    return torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

def cosine_schedule(opt, total_steps, warmup_frac=0.03):
    w = max(1, int(total_steps * warmup_frac))
    f = lambda s: (s + 1) / w if s < w else 0.5 * (1 + math.cos(math.pi * (s - w) / max(1, total_steps - w)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def val_macro_auroc(y, s):
    from sklearn.metrics import roc_auc_score
    return float(np.mean([roc_auc_score(y[:, c], s[:, c]) for c in range(C) if 0 < y[:, c].sum() < len(y)]))

def train_model(cfg, exp_dir, model):
    loss_fn = AsymmetricLoss(**cfg["asl"])
    train_dl, val_dl = loader(df_train, cfg, True), loader(df_val, cfg, False, cfg["batch_size"] * 2)
    yv = df_val[CLASSES].values
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    plan = [("warmup", i) for i in range(cfg["warmup_epochs"])] + [("finetune", i) for i in range(cfg["finetune_epochs"])]
    best, best_epoch, bad, ema, phase_now, t0, ep_times, truncated = -1.0, 0, 0, None, None, time.time(), [], False
    for ep, (phase, _) in enumerate(plan, 1):
        if ep_times and (time.time() - t0 + max(ep_times)) / 3600 > cfg["time_budget_h"]:
            print("Stopping: next epoch would exceed the time budget -> truncated=true"); truncated = True
            break
        if phase != phase_now:
            phase_now, bad = phase, 0
            opt = configure_phase(model, cfg, phase)
            n_ep = cfg["warmup_epochs"] if phase == "warmup" else cfg["finetune_epochs"]
            sched = cosine_schedule(opt, n_ep * len(train_dl))
            if phase == "finetune" and cfg["use_ema"]:
                from timm.utils import ModelEmaV2
                ema = ModelEmaV2(model, decay=cfg["ema_decay"])
        if USE_AMP:
            torch.cuda.reset_peak_memory_stats()
        model.train()
        te, run_loss = time.time(), 0.0
        for i, (x, y) in enumerate(train_dl):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                ema.update(model)
            run_loss += loss.item()
            if i % 500 == 0:
                print(f"  epoch {ep} step {i}/{len(train_dl)} loss {loss.item():.4f}", flush=True)
        eval_model = ema.module if ema is not None else model                 # EMA weights are evaluated (M09)
        zv = predict_logits(eval_model, val_dl)
        auc = val_macro_auroc(yv, zv)
        ep_times.append(time.time() - te)
        row = dict(epoch=ep, phase=phase, train_loss=run_loss / len(train_dl),
                   val_loss=float(loss_fn(torch.as_tensor(zv), torch.as_tensor(yv))), val_macro_auroc=auc,
                   epoch_seconds=ep_times[-1], peak_gpu_mem_mb=torch.cuda.max_memory_allocated() / 2 ** 20 if USE_AMP else np.nan)
        log = f"{exp_dir}/train_log.csv"
        pd.DataFrame([row]).to_csv(log, mode="a", header=not os.path.exists(log), index=False)
        print(f"Epoch {ep} [{phase}] loss {row['train_loss']:.4f} | val macro AUROC {auc:.4f} | {ep_times[-1] / 60:.1f} min", flush=True)
        if auc > best:                                                        # strict '>' keeps the earlier epoch on ties
            best, best_epoch, bad = auc, ep, 0
            torch.save(eval_model.state_dict(), f"{exp_dir}/checkpoint_best.pt")
        elif phase == "finetune":
            bad += 1
            if bad >= cfg["patience"]:
                print("Early stopping."); break
    model.load_state_dict(torch.load(f"{exp_dir}/checkpoint_best.pt", map_location=DEVICE))
    return model, dict(best_val_macro_auroc=best, best_epoch=best_epoch, epochs_trained=len(ep_times),
                       truncated=truncated, train_wall_clock_h=(time.time() - t0) / 3600)

def run_stage(cfg):
    """Train one model with SEED and export val/test logits. Skips if the seed-42 logits already exist."""
    existing = find_run_dir(cfg["model_id"])
    if existing:
        print(f"{cfg['model_id']}: logits already at {existing} - skipping."); return
    exp_dir = os.path.join(RUNS_DIR, run_folder_name(cfg['model_id']))
    os.makedirs(exp_dir, exist_ok=True)
    set_determinism()
    model = build_model(cfg)
    model, info = train_model(cfg, exp_dir, model)
    for part, frame in (("val", df_val), ("test", df_test)):
        z = predict_logits(model, loader(frame, cfg, False, cfg["batch_size"] * 2))
        np.savez(f"{exp_dir}/logits_{part}.npz", image_index=frame["Image Index"].to_numpy(dtype=str),
                 patient_id=frame["Patient ID"].to_numpy(dtype=np.int64), logits=z,
                 labels=frame[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
                 split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, model_id=cfg["model_id"], seed=SEED, eval_set=part)
    import timm, transformers
    write_json(f"{exp_dir}/config.json", dict(
        model_id=cfg["model_id"], seed=SEED, split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, protocol_version=PROTOCOL_VERSION,
        model_cfg=cfg, train_info=info, checkpoint_sha256=sha256_file(f"{exp_dir}/checkpoint_best.pt"),
        software=dict(torch=torch.__version__, timm=timm.__version__, transformers=transformers.__version__),
        gpu=torch.cuda.get_device_name(0) if USE_AMP else "cpu"))
    print(f"{cfg['model_id']}: done ({info['train_wall_clock_h']:.2f} h, best epoch {info['best_epoch']})")

def benchmark(cfg):
    """Forward-pass latency per image (fp16, no TTA). Weights are random: latency does not depend on their values."""
    assert USE_AMP, "benchmark needs a GPU"
    model = build_model(cfg, pretrained=False).eval()
    res = dict(model_id=cfg["model_id"], gpu=torch.cuda.get_device_name(0), torch=torch.__version__, amp_dtype="fp16",
               scope="model forward pass on a preloaded tensor, no TTA", weights="random (same architecture)",
               warmup_iters=WARMUP_ITERS, timed_iters=TIMED_ITERS, repeats=TIMING_REPEATS,
               total_params=sum(p.numel() for p in model.parameters()), source="session T benchmark")
    for bs in LATENCY_BATCH_SIZES:
        x = torch.randn(bs, 3, cfg["img_size"], cfg["img_size"], device=DEVICE)
        meds = []
        for _ in range(TIMING_REPEATS):
            with torch.inference_mode(), autocast():
                for _ in range(WARMUP_ITERS):
                    model(x)
                torch.cuda.synchronize()
                ms = []
                for _ in range(TIMED_ITERS):
                    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
                    s.record(); model(x); e.record(); torch.cuda.synchronize()
                    ms.append(s.elapsed_time(e) / bs)
            meds.append(float(np.median(ms)))
        res[f"bs{bs}_median_ms_per_image"], res[f"bs{bs}_repeat_medians"] = float(np.median(meds)), meds
    write_json(f"{OUT_DIR}/timing_{cfg['model_id']}.json", res)
    print(cfg["model_id"], {k: round(v, 2) for k, v in res.items() if k.endswith("median_ms_per_image")})
    del model; torch.cuda.empty_cache()

## Run

Set `RUN_CNN = True` for M08, `RUN_VIT = True` for M09, or enable both to run sequentially. Set `RUN_BENCH = True` only when you want the separate GPU timing benchmark. Outputs are written under `/kaggle/working/single_model/output/results_s42/` (override with `CXR_OUT_DIR`); model run directories are placed in its `runs/` subdirectory.

The runner skips a matching seed/model run found in the output or attached Kaggle inputs. To train a fresh run, avoid attaching an existing matching run folder. Download the complete `results_s42/` directory to preserve checkpoints, logits, split, and timing artifacts.


In [ ]:
if RUN_CNN:
    run_stage(CNN_CFG)
if RUN_VIT:
    run_stage(VIT_CFG)
if RUN_BENCH:
    for cfg in (CNN_CFG, VIT_CFG):
        benchmark(cfg)